# 02. Build Audit Cases And Retrieval Index

이 노트북은 Track B의 입력 데이터를 준비합니다.

- Ko/En audit case를 validation split에서 샘플링합니다.
- 각 case는 target token 단위로 구성됩니다.
- training split에서 raw→norm retrieval index를 만듭니다.

주요 출력:

- `outputs/context_rag/audit_cases.csv`
- `outputs/context_rag/retrieval_index.csv`

`audit_cases.csv`는 이후 metadata generation, S0/S1/S2 normalizer, judge 평가가 모두 공유하는 기준 case 파일입니다.


## 실험 설정

프로젝트 루트를 고정하고 사용할 Hugging Face 데이터셋 이름을 정의합니다. 이 노트북은 `scripts/`를 subprocess로 실행하지 않고 `src/lexnorm` 함수를 직접 호출합니다. 같은 기능은 `scripts/02_build_cases.py`, `scripts/02_build_retrieval_index.py`로도 command 실행 가능합니다.


In [ ]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.utils import sync_to_drive

from lexnorm.data import build_audit_cases, build_retrieval_index

DATASET = "weerayut/multilexnorm2026-dev-pub"
CASES_CSV = "outputs/context_rag/audit_cases.csv"
INDEX_CSV = "outputs/context_rag/retrieval_index.csv"


## audit case 생성

Korean 1000개, English 300개 target-token case를 만들고 `audit_cases.csv`로 저장합니다.


In [ ]:
audit_df = build_audit_cases(
    dataset_name=DATASET,
    output=CASES_CSV,
    n_ko=1000,
    n_en=300,
    split="validation",
)
print("saved", CASES_CSV, len(audit_df))
display(audit_df.head())
sync_to_drive(CASES_CSV)
if len(audit_df) and "sample_group" in audit_df:
    display(audit_df["sample_group"].value_counts().to_frame("count"))


## retrieval index 생성

Ko/En train split에서 raw token별 normalized candidate와 예문을 모아 RAG용 index를 만듭니다.


In [ ]:
index_df = build_retrieval_index(
    dataset_name=DATASET,
    output=INDEX_CSV,
    langs=["ko", "en"],
    split="train",
)
print("saved", INDEX_CSV, len(index_df))
display(index_df.head())
sync_to_drive(INDEX_CSV)
